<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/stage_06_00_model_training_plan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_06 – Model Training – Objetivo y Modelos**


# **1. Plan de entrenamiento**

En este notebook se definirá el plan de entrenamiento para los modelos aplicados al problema de **clasificación secuencial T2**, manteniendo un esquema experimental consistente, comparable y alineado con el flujo metodológico de *Machine Learning for Trading*: definición del problema supervisado, fijación del objetivo predictivo, diseño del modelo, evaluación fuera de muestra y control del overfitting.

El objetivo de esta etapa es comparar múltiples arquitecturas bajo exactamente el mismo problema predictivo, de modo que las diferencias observadas en desempeño respondan al modelo utilizado o a la regla de decisión aplicada, y no a cambios en el target, los datos o el pipeline experimental.

**Objetivo predictivo**

En esta etapa, el objetivo predictivo queda definido por un conjunto de elementos que deben permanecer fijos para todos los modelos comparados:

* los **targets estructurales**
* el **dataset base**
* los **splits temporales**
* el **universo de observaciones y regímenes**
* el **conjunto de features**
* el **window size**
* el **pipeline de construcción de ventanas**

Esto garantiza que todos los modelos sean evaluados sobre el mismo problema supervisado y bajo las mismas condiciones experimentales.

**Targets estructurales**

El enfoque será exclusivamente de **clasificación multiclase T2**, utilizando los siguientes targets estructurales seleccionados:

* `t2_p40_h30`
* `t2_p40_h60`
* `t2_p50_h30`

Estos targets representan una clasificación direccional con umbral, construida a partir del movimiento futuro del precio respecto de percentiles definidos sobre la distribución de desplazamientos observados.

En este esquema, el **target estructural** define qué señal se desea aprender. Por lo tanto, todos los modelos comparados deberán entrenarse sobre estos mismos targets para que la comparación sea metodológicamente válida.

**Dataset base**

Todos los modelos deberán entrenarse y evaluarse sobre el mismo dataset base, respetando:

* los mismos splits
* las mismas observaciones
* el mismo universo de regímenes
* el mismo orden temporal
* la misma construcción de muestras

Esto permite evitar que diferencias en desempeño se deban a cambios en la base de datos o en la muestra utilizada.

**Window size**

El modelado se realizará en formato secuencial, utilizando ventanas históricas de longitud fija como contexto de entrada.

En esta etapa se trabajará con un único tamaño de ventana:

* `30`

La decisión de fijar un único `window_size` busca eliminar una fuente adicional de variabilidad experimental y concentrar la comparación en las diferencias entre targets, modelos y reglas de decisión.

**Features**

Para que la comparación entre modelos sea justa, todos deberán utilizar el mismo conjunto de features definido para esta etapa.

```python
features_t2 = [
    "ema_60",
    "roc_60",
    "roc_30",
    "stoch_k_30",
    "mom_5",
    "atr_norm_10",
    "macd",
]

```

Estas features constituyen la representación de entrada común para todos los modelos en esta fase. La exploración de nuevos conjuntos de variables podrá realizarse en una etapa posterior, pero no debe mezclarse con la comparación principal entre arquitecturas.

**Enfoque de predicción**

En esta etapa el enfoque será únicamente:

* **seq2one**

Es decir, cada secuencia de entrada producirá una única predicción correspondiente a la etiqueta T2 asociada al instante final de la ventana.

No se trabajará con un enfoque **seq2seq** en este stage.

Esta formulación es coherente con el objetivo del proyecto: evaluar si una secuencia corta de contexto reciente permite anticipar una señal estructural útil en el mercado intradía.

**Qué se mantiene fijo**

Para todos los modelos comparados, deberán permanecer fijos:

* los targets estructurales
* el dataset base
* los splits temporales
* el universo de regímenes
* el conjunto de features
* el `window_size = 30`
* el pipeline de entrenamiento y evaluación

**Qué puede cambiar**

Las dimensiones que sí pueden variar en esta etapa son:

* el modelo
* la regla de decisión aplicada sobre las probabilidades del modelo
* el `decision_threshold`

Los modelos a comparar son:

* XGBoost
* LightGBM
* GRU
* Transformer
* Logistic Regression
* Random Forest

**Regla de decisión**

La regla de decisión no forma parte del target, sino de la capa operativa que transforma una predicción en una señal accionable.

Por esta razón, configuraciones como:

* `decision_threshold = 0.30`
* `decision_threshold = 0.35`

no definen problemas predictivos distintos, sino distintas formas de explotar operativamente un mismo modelo entrenado sobre un mismo target estructural.

En consecuencia:

* el **target** define qué se quiere predecir
* el **modelo** define cómo se aprende esa señal
* el **decision threshold** define cuándo esa predicción se convierte en operación

**Tipo de problema**

El problema queda definido formalmente como:

* **aprendizaje supervisado**
* **clasificación multiclase**
* **modelado secuencial seq2one**
* **predicción intradía sobre datos minuto a minuto del MNQ**

**Objetivo del stage**

El objetivo de este stage será comparar de forma sistemática múltiples arquitecturas bajo el mismo problema predictivo estructural, analizando:

* diferencias entre targets T2 seleccionados
* capacidad predictiva de distintas arquitecturas
* robustez fuera de muestra
* sensibilidad de la capa operativa a distintos `decision_threshold`
* equilibrio entre desempeño predictivo y utilidad operativa

El propósito no es únicamente identificar el mejor score puntual, sino separar con claridad tres niveles del sistema:

* el nivel **estructural**, definido por el target
* el nivel **predictivo**, definido por el modelo
* el nivel **operativo**, definido por la regla de decisión

## **Resumen**

**1. Qué predice el modelo**

El modelo aprende a predecir una **clase T2**:

* `+1` → movimiento alcista significativo
* `0` → no movimiento relevante
* `-1` → movimiento bajista significativo

Es decir, **clasificación multiclase** basada en el comportamiento futuro del precio bajo el umbral definido.

---

**2. Qué devuelve el modelo realmente**

No solo devuelve la clase, sino también **probabilidades**:

```python
proba = {
    -1: 0.25,
     0: 0.50,
     1: 0.25
}
```

Y de ahí:

* clase predicha → `argmax`
* confidence → `max(proba)`

---

**3. Dónde entra la decisión operativa**

Ahí es donde separas bien las cosas:

El modelo **NO decide operar**.
Solo estima probabilidades.

La decisión viene después, por ejemplo:

* si `proba(+1) ≥ 0.30` → BUY
* si `proba(-1) ≥ 0.30` → SELL
* si no → NO OPERAR

---

**4. Interpretación correcta**

* El modelo aprende:
  → “qué tan probable es cada escenario (+1, 0, -1)”

* La regla de decisión define:
  → “cuándo esa probabilidad es suficientemente fuerte para operar”

---

**5. Resumen en una línea**

Sí:

> El modelo predice `+1 / 0 / -1`, y luego usas la **confidence (probabilidad)** para decidir si operar o no.

---

Ese desacople es clave, porque te permite:

* comparar modelos sobre el mismo problema
* cambiar la agresividad operativa sin reentrenar
* analizar señal vs ejecución por separado

# **2. Estructura dimensional del problema de predicción**

## **2.1. Ventana de entrada $X$**

Las ventanas de entrada tienen la siguiente dimensión:


$$
L \times N
$$

donde:

- $L = 30 $ representa la longitud fija de la ventana histórica (window_size).
- $N = 7$ corresponde al número de variables predictoras (features).

Por lo tanto, cada muestra de entrada tiene dimensión:

$$ X ∈ R^{30 × 7} $$

Cada ventana contiene información temporal reciente del mercado, resumida mediante indicadores técnicos y variables derivadas.

## **2.2 Ventana de salida $y$**

Se utilizará un enfoque seq2one, donde cada ventana de entrada produce una única predicción.

- **Salida / Target (Y)**

  $$
  Y ∈ {-1, 0, 1}
  $$

  Este valor corresponde a la clase T2 asociada al comportamiento futuro del precio:

  * `+1` → movimiento alcista relevante
  * `0` → no movimiento significativo
  * `-1` → movimiento bajista relevante

## **2.3. Horizontes de predicción**


El modelo se entrenará sobre distintos horizontes de predicción:

* `H = 30`
  * `t2_p40_h30`
  * `t2_p50_h30`

* `H = 60`
  * `t2_p40_h60`

Cada horizonte define el período futuro sobre el cual se evalúa el movimiento del precio para construir el target.

Esto implica que:

* el input (`X`) es el mismo
* el target (`Y`) cambia según el horizonte


## **2.4 Interpretación del modelo**



El modelo aprende una función del tipo:

$$
f: R^{30 × 7} → {−1, 0, 1}
$$

Es decir:

> a partir de una ventana histórica de 30 pasos y 7 features, el modelo predice una clase que representa el movimiento futuro del precio.

En la práctica, el modelo estima:

$$
P(Y = -1), P(Y = 0), P(Y = 1)
$$

y la clase final se obtiene como:

$$
argmax(P)
$$


## **2.5. Interpretación operativa**

Aunque el modelo predice una clase, su salida real son probabilidades.

Estas probabilidades permiten:

- estimar la confidence
- aplicar reglas de decisión (thresholds)
- convertir predicciones en señales operativas

## **Idea central**



El modelo no predice directamente el retorno ni la operación, sino:

> una **clasificación probabilística del comportamiento futuro del precio**, condicionada al horizonte H.

```
Cambio clave respecto a tu versión anterior:

- antes: regresión (`Y ∈ R`)  
- ahora: clasificación (`Y ∈ {-1,0,1}`)

- antes: múltiples ventanas  
- ahora: **una sola (30)**

- antes: 5 features  
- ahora: **7 features**

- antes: output escalar  
- ahora: **distribución de probabilidad**
```

# **3. Modelos a entrenar**


En este stage se evaluarán distintas familias de modelos para el problema de **clasificación secuencial seq2one (T2)**, con el objetivo de comparar su capacidad predictiva bajo exactamente el mismo problema estructural.

Se busca un balance entre:

* robustez en datos financieros (alta presencia de ruido)
* capacidad de capturar relaciones no lineales
* capacidad de explotar (o no) la estructura temporal
* complejidad vs generalización


## **3.1 Enfoque general**

Se trabajará con dos enfoques complementarios:

**1) Modelos tabulares (baseline)**

Estos modelos **no modelan explícitamente la secuencia**, ya que operan sobre la representación de entrada ya construida (ventanas transformadas en features).

Su objetivo es:

* establecer un baseline sólido
* verificar si la estructura secuencial aporta valor real
* reducir el riesgo de sobreajuste

Esto es consistente con la práctica en datos financieros, donde modelos simples y robustos suelen ser difíciles de superar.

Modelos incluidos:

* Logistic Regression
* Random Forest
* Gradient Boosting:
  * XGBoost
  * LightGBM

Los métodos de boosting son especialmente relevantes, ya que suelen ofrecer muy buen desempeño en datos tabulares estructurados.

---

**2) Modelos secuenciales (deep learning)**

Estos modelos explotan explícitamente la dimensión temporal de los datos, utilizando la estructura `(30 × 7)` como secuencia.

Su objetivo es:

* capturar dependencias temporales
* modelar dinámicas no lineales en la evolución de las features
* evaluar si la secuencia aporta señal adicional

Modelos incluidos:

- GRU

  * Arquitectura recurrente eficiente
  * Menor complejidad que LSTM
  * Mejor relación bias–variance en muchos casos
  * Modelo secuencial principal

- Transformer (Encoder)

  * Modela relaciones globales dentro de la secuencia
  * No depende de recurrencia
  * Puede capturar interacciones más complejas entre pasos temporales


## **3.2. Justificación de la selección**

La selección cubre distintos niveles de complejidad y supuestos:

| Tipo de modelo     | Rol en el experimento        |
|------------------|-----------------------------|
| Logistic         | baseline interpretable       |
| Random Forest    | baseline no lineal           |
| Boosting         | baseline fuerte (tabular)    |
| GRU              | secuencial clásico           |
| Transformer      | secuencial avanzado          |

Esto permite responder preguntas clave:

* ¿La secuencia aporta valor real frente a modelos tabulares?
* ¿Los modelos complejos mejoran la señal o solo ajustan ruido?
* ¿Existe señal explotable bajo el esquema T2 definido?

## **3.3. Enfoque experimental**

Todos los modelos se entrenarán bajo un esquema estrictamente consistente:

* mismos targets estructurales
* mismo `window_size = 30`
* mismas features
* mismos splits temporales
* mismo dataset base
* mismo pipeline de construcción de ventanas

Las únicas dimensiones que varían son:

* el modelo
* la regla de decisión (threshold)

Esto garantiza que la comparación entre modelos sea válida y metodológicamente correcta.

## **Idea central**


Todos los modelos intentan resolver exactamente el mismo problema:

> estimar la probabilidad de cada clase T2 (`-1`, `0`, `+1`) a partir de una ventana histórica.

La comparación entre ellos permitirá identificar qué tipo de arquitectura captura mejor la señal presente en los datos.

# **4. Comparación entre etiqueta real y predicción**

En este stage, la evaluación del modelo se realiza exclusivamente bajo el esquema **seq2one**.

Cada ventana histórica de entrada produce una única predicción asociada a una sola etiqueta futura del target T2.

Para cada muestra \(t\), la entrada del modelo es:

$$
X_t \in \mathbb{R}^{30 \times 7}
$$

donde:

* `30` es la longitud fija de la ventana histórica
* `7` es el número de features utilizadas en cada paso temporal

A partir de esta ventana, el modelo genera una única predicción de clase:

$$
\hat{y}_t \in \{-1,0,1\}
$$

donde:

* `+1` representa un movimiento alcista relevante
* `0` representa ausencia de movimiento significativo
* `-1` representa un movimiento bajista relevante

Esta predicción se compara directamente contra la etiqueta real observada:

$$
y_t \in \{-1,0,1\}
$$

La comparación se realiza entonces bajo la siguiente lógica:

* **una ventana de entrada**
* **una única predicción**
* **una única etiqueta real asociada**

No se predice una secuencia futura completa ni una trayectoria de precios.
Cada ventana resume el contexto reciente del mercado y el modelo debe emitir una sola clasificación sobre el comportamiento futuro del precio.


## **4.1 Interpretación del esquema seq2one**


Bajo este enfoque, el modelo aprende una función del tipo:

$$
f:\mathbb{R}^{30 \times 7} \rightarrow \{-1,0,1\}
$$

Es decir, transforma una ventana temporal multivariada en una única clase objetivo.

En términos prácticos, el modelo no solo entrega una clase final, sino también una distribución de probabilidad sobre las tres clases:

$$
P(Y=-1 \mid X_t), \; P(Y=0 \mid X_t), \; P(Y=1 \mid X_t)
$$

La clase predicha corresponde a la de mayor probabilidad, mientras que estas probabilidades también permiten calcular la **confidence** y aplicar posteriormente la regla de decisión operativa.

Este planteamiento es consistente con el diseño actual del problema T2, donde el interés no está en predecir toda una trayectoria futura, sino en clasificar el comportamiento futuro del precio en una categoría discreta y operativamente interpretable.

# **5. Métricas y evaluación del modelo**

## **5.1 Enfoque de evaluación**

Dado que el problema es de **clasificación multiclase seq2one**, cada muestra produce:

* una predicción de clase: $\hat{y}_t \in \{-1,0,1\}$
* una etiqueta real: $y_t \in \{-1,0,1\}$

Las métricas se calculan sobre el conjunto completo de muestras de cada split.

La comparación es:

* **predicción vs etiqueta real**
* no hay valores continuos
* no hay error de magnitud

Adicionalmente, el modelo genera probabilidades por clase:

$$
P(Y=-1),\; P(Y=0),\; P(Y=1)
$$

Estas probabilidades no se utilizan directamente en las métricas de este stage, pero serán clave en la etapa operativa para definir reglas de decisión.


## **5.2 Métricas de Machine Learning**

Las métricas se agrupan en tres niveles:

**1. Métricas principales (criterio de selección)**

Estas métricas determinan qué modelos continúan:

* **Balanced Accuracy**

  * Corrige el desbalance de clases
  * Métrica principal para el problema T2

* **F1 Score (macro o weighted)**

  * Balance entre precisión y recall
  * Complementa la balanced accuracy

---

**2. Métricas complementarias**

Se reportan para interpretación adicional:

* **Accuracy**
* **Precision (macro / weighted)**
* **Recall (macro / weighted)**

---

**3. Métricas diagnósticas**

Se utilizan para análisis interno del comportamiento del modelo:

* matriz de confusión
* distribución de clases predichas
* comparación contra baseline (naive)


## **5.3 Baseline de referencia**


Todos los modelos deben compararse contra un baseline simple:

* **Naive classifier**

  * predice siempre la clase más frecuente

Esto permite verificar si el modelo realmente captura señal o simplemente replica la distribución de clases.


## **5.4 Uso de los splits (criterio de evaluación)**


Se mantiene el esquema estándar del workflow:

* **TRAIN**

  * usado exclusivamente para entrenamiento

* **VALID**

  * usado para:
    * comparar modelos
    * seleccionar hiperparámetros
    * decidir qué modelos continúan

* **TEST**

  * usado una única vez
  * evaluación final del modelo seleccionado

El conjunto TEST no se utiliza durante entrenamiento ni tuning, evitando leakage y asegurando evaluación fuera de muestra.


## **5.5 Protocolo de evaluación**


Para cada modelo:

1. entrenar en TRAIN
2. evaluar en VALID
3. calcular métricas principales
4. comparar contra baseline
5. seleccionar candidatos

El conjunto TEST se utiliza únicamente al final del stage.


## **5.6 Relación con la decisión operativa**

En este stage no se evalúa directamente la performance operativa.

El modelo se evalúa únicamente como clasificador.

La conversión de probabilidades en decisiones (por ejemplo, mediante `decision_threshold`) se realizará en una etapa posterior.

Esto permite separar claramente:

* **predicción estadística** (este stage)
* **decisión operativa** (stage siguiente)

## **5.7 Métricas económicas (etapa posterior)**


Las métricas económicas no forman parte de este stage.

Una vez seleccionados los modelos candidatos, se evaluarán utilizando:

* retorno esperado (EV)
* métricas riesgo-retorno
* drawdown

Esto permite traducir el desempeño estadístico en impacto operativo real.

## **5.8 Jerarquía de métricas**




**Criterio de selección:**

* Balanced Accuracy
* F1 Score

**Métricas complementarias:**

* Accuracy
* Precision
* Recall

**Diagnóstico:**

* matriz de confusión
* comparación baseline vs modelo

# **6. Función general de métricas**

Para estandarizar la evaluación de todos los modelos del problema de clasificación T2, se implementa un módulo reutilizable de Python, importable desde notebooks y scripts de entrenamiento.

La función principal recibe como entrada las etiquetas reales y las clases predichas por el modelo, y devuelve un bloque unificado de resultados de evaluación.

En particular, incluye:

* métricas principales de clasificación
* métricas complementarias
* comparación contra baseline naive
* matriz de confusión
* distribución de clases reales y predichas

Además, el módulo permite incorporar metadata del experimento, como el nombre del modelo, el split evaluado y el target utilizado.

Este diseño permite evaluar de forma consistente modelos tabulares y secuenciales bajo el mismo criterio experimental, asegurando comparabilidad entre arquitecturas, targets y splits.

La idea central es que todos los modelos, independientemente de su tipo, sean evaluados con exactamente la misma lógica y bajo el mismo protocolo.

In [10]:
classification_metrics_py = '''
"""
classification_metrics.py

Utilidades de métricas para problemas de clasificación seq2one
aplicados al proyecto MNQ T2.

Diseñado para ser importado desde notebooks o scripts de entrenamiento.

Métricas principales:
- balanced_accuracy
- f1_macro
- f1_weighted

Métricas complementarias:
- accuracy
- precision_macro
- precision_weighted
- recall_macro
- recall_weighted

Diagnóstico:
- confusion_matrix
- class distribution real/predicha
- baseline naive (clase más frecuente en y_true)

Autor: OpenAI / Proyecto MNQ
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, Optional

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


@dataclass
class ClassificationMetricsConfig:
    """
    Configuración para el cálculo de métricas de clasificación.

    Parámetros
    ----------
    average_macro : str
        Tipo de promedio usado para métricas macro.
    average_weighted : str
        Tipo de promedio usado para métricas weighted.
    zero_division : int
        Valor a usar cuando una métrica presenta división por cero.
    include_confusion_matrix : bool
        Si True, incluye matriz de confusión en la salida.
    include_class_distribution : bool
        Si True, incluye distribución de clases reales y predichas.
    include_naive_baseline : bool
        Si True, calcula baseline naive basado en la clase más frecuente.
    """
    average_macro: str = "macro"
    average_weighted: str = "weighted"
    zero_division: int = 0
    include_confusion_matrix: bool = True
    include_class_distribution: bool = True
    include_naive_baseline: bool = True


def _to_1d_numpy(y: Iterable[Any], name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray 1D.

    Parámetros
    ----------
    y : iterable
        Etiquetas reales o clases predichas.
    name : str
        Nombre de la variable, solo para mensajes de error.

    Retorna
    -------
    np.ndarray
        Vector 1D.
    """
    arr = np.asarray(y)

    if arr.ndim == 0:
        raise ValueError(f"{name} no puede ser escalar; se esperaba un vector 1D.")
    if arr.ndim > 1:
        arr = arr.reshape(-1)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío.")

    return arr


def _validate_labels(labels: np.ndarray) -> np.ndarray:
    """
    Valida que el vector de labels no esté vacío ni contenga duplicados.

    Parámetros
    ----------
    labels : np.ndarray
        Vector de clases.

    Retorna
    -------
    np.ndarray
        Vector validado.
    """
    if labels.size == 0:
        raise ValueError("El parámetro 'labels' no puede estar vacío.")

    unique_labels = np.unique(labels)
    if unique_labels.size != labels.size:
        raise ValueError(
            "El parámetro 'labels' contiene clases duplicadas. "
            "Debe especificarse un conjunto ordenado de clases sin repeticiones."
        )

    return labels


def _build_naive_predictions(y_true: np.ndarray) -> np.ndarray:
    """
    Construye un baseline naive que predice siempre la clase más frecuente en y_true.

    Parámetros
    ----------
    y_true : np.ndarray
        Etiquetas reales.

    Retorna
    -------
    np.ndarray
        Predicción naive.
    """
    classes, counts = np.unique(y_true, return_counts=True)
    majority_class = classes[np.argmax(counts)]
    return np.full(shape=y_true.shape, fill_value=majority_class, dtype=y_true.dtype)


def _compute_core_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    *,
    average_macro: str = "macro",
    average_weighted: str = "weighted",
    zero_division: int = 0,
) -> Dict[str, float]:
    """
    Calcula métricas principales y complementarias.

    Parámetros
    ----------
    y_true : np.ndarray
        Etiquetas reales.
    y_pred : np.ndarray
        Clases predichas.
    average_macro : str
        Promedio usado para métricas macro.
    average_weighted : str
        Promedio usado para métricas weighted.
    zero_division : int
        Valor ante divisiones por cero.

    Retorna
    -------
    dict
        Diccionario de métricas.
    """
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average=average_macro,
                zero_division=zero_division,
            )
        ),
        "f1_weighted": float(
            f1_score(
                y_true,
                y_pred,
                average=average_weighted,
                zero_division=zero_division,
            )
        ),
        "precision_macro": float(
            precision_score(
                y_true,
                y_pred,
                average=average_macro,
                zero_division=zero_division,
            )
        ),
        "precision_weighted": float(
            precision_score(
                y_true,
                y_pred,
                average=average_weighted,
                zero_division=zero_division,
            )
        ),
        "recall_macro": float(
            recall_score(
                y_true,
                y_pred,
                average=average_macro,
                zero_division=zero_division,
            )
        ),
        "recall_weighted": float(
            recall_score(
                y_true,
                y_pred,
                average=average_weighted,
                zero_division=zero_division,
            )
        ),
    }


def compute_classification_metrics(
    y_true: Iterable[Any],
    y_pred: Iterable[Any],
    *,
    model_name: Optional[str] = None,
    split: Optional[str] = None,
    target: Optional[str] = None,
    labels: Optional[Iterable[Any]] = None,
    config: Optional[ClassificationMetricsConfig] = None,
) -> Dict[str, Any]:
    """
    Calcula métricas estandarizadas para clasificación seq2one.

    Parámetros
    ----------
    y_true : iterable
        Etiquetas reales.
    y_pred : iterable
        Clases predichas por el modelo.
    model_name : str, opcional
        Nombre del modelo.
    split : str, opcional
        Split evaluado: train / valid / test.
    target : str, opcional
        Nombre del target.
    labels : iterable, opcional
        Orden explícito de clases para la matriz de confusión.
        Si es None, se usa la unión ordenada de y_true e y_pred.
    config : ClassificationMetricsConfig, opcional
        Configuración del cálculo.

    Retorna
    -------
    dict
        Diccionario con:
        - metrics
        - baseline naive
        - matriz de confusión
        - distribuciones de clases
    """
    cfg = config or ClassificationMetricsConfig()

    y_true_arr = _to_1d_numpy(y_true, "y_true")
    y_pred_arr = _to_1d_numpy(y_pred, "y_pred")

    if y_true_arr.shape[0] != y_pred_arr.shape[0]:
        raise ValueError(
            f"y_true e y_pred deben tener la misma longitud. "
            f"Recibido: {y_true_arr.shape[0]} vs {y_pred_arr.shape[0]}"
        )

    if labels is None:
        final_labels = np.unique(np.concatenate([y_true_arr, y_pred_arr]))
    else:
        final_labels = _validate_labels(np.asarray(list(labels)))

    metrics = {
        "model": model_name,
        "split": split,
        "target": target,
        "n_samples": int(len(y_true_arr)),
        **_compute_core_metrics(
            y_true_arr,
            y_pred_arr,
            average_macro=cfg.average_macro,
            average_weighted=cfg.average_weighted,
            zero_division=cfg.zero_division,
        ),
    }

    if cfg.include_naive_baseline:
        y_pred_naive = _build_naive_predictions(y_true_arr)
        naive_metrics = _compute_core_metrics(
            y_true_arr,
            y_pred_naive,
            average_macro=cfg.average_macro,
            average_weighted=cfg.average_weighted,
            zero_division=cfg.zero_division,
        )
        metrics.update({
            "accuracy_naive": naive_metrics["accuracy"],
            "balanced_accuracy_naive": naive_metrics["balanced_accuracy"],
            "f1_macro_naive": naive_metrics["f1_macro"],
            "f1_weighted_naive": naive_metrics["f1_weighted"],
            "precision_macro_naive": naive_metrics["precision_macro"],
            "precision_weighted_naive": naive_metrics["precision_weighted"],
            "recall_macro_naive": naive_metrics["recall_macro"],
            "recall_weighted_naive": naive_metrics["recall_weighted"],
            "balanced_accuracy_gain_vs_naive": (
                metrics["balanced_accuracy"] - naive_metrics["balanced_accuracy"]
            ),
            "f1_macro_gain_vs_naive": (
                metrics["f1_macro"] - naive_metrics["f1_macro"]
            ),
            "f1_weighted_gain_vs_naive": (
                metrics["f1_weighted"] - naive_metrics["f1_weighted"]
            ),
        })

    result: Dict[str, Any] = {"metrics": metrics}

    if cfg.include_confusion_matrix:
        cm = confusion_matrix(y_true_arr, y_pred_arr, labels=final_labels)
        result["confusion_matrix"] = cm.tolist()
        result["confusion_matrix_labels"] = final_labels.tolist()

    if cfg.include_class_distribution:
        true_classes, true_counts = np.unique(y_true_arr, return_counts=True)
        pred_classes, pred_counts = np.unique(y_pred_arr, return_counts=True)

        result["class_distribution_true"] = {
            str(cls): int(cnt) for cls, cnt in zip(true_classes, true_counts)
        }
        result["class_distribution_pred"] = {
            str(cls): int(cnt) for cls, cnt in zip(pred_classes, pred_counts)
        }

    return result


def metrics_to_flat_dict(metrics_result: Dict[str, Any]) -> Dict[str, Any]:
    """
    Aplana la salida de compute_classification_metrics para convertirla
    fácilmente en DataFrame.

    Parámetros
    ----------
    metrics_result : dict
        Salida de compute_classification_metrics.

    Retorna
    -------
    dict
        Diccionario plano solo con métricas y metadata.
    """
    if "metrics" not in metrics_result:
        raise ValueError("El diccionario recibido no contiene la clave 'metrics'.")

    return dict(metrics_result["metrics"])


def print_classification_report_block(metrics_result: Dict[str, Any]) -> None:
    """
    Imprime un bloque compacto y legible con las métricas principales.

    Parámetros
    ----------
    metrics_result : dict
        Salida de compute_classification_metrics.
    """
    if "metrics" not in metrics_result:
        raise ValueError("El diccionario recibido no contiene la clave 'metrics'.")

    m = metrics_result["metrics"]

    title = (
        f"CLASSIFICATION REPORT | "
        f"model={m.get('model')} | split={m.get('split')} | target={m.get('target')}"
    )
    print("=" * len(title))
    print(title)
    print("=" * len(title))
    print(f"n_samples               : {m.get('n_samples')}")
    print(f"balanced_accuracy       : {m.get('balanced_accuracy'):.6f}")
    print(f"f1_macro                : {m.get('f1_macro'):.6f}")
    print(f"f1_weighted             : {m.get('f1_weighted'):.6f}")
    print(f"accuracy                : {m.get('accuracy'):.6f}")
    print(f"precision_macro         : {m.get('precision_macro'):.6f}")
    print(f"precision_weighted      : {m.get('precision_weighted'):.6f}")
    print(f"recall_macro            : {m.get('recall_macro'):.6f}")
    print(f"recall_weighted         : {m.get('recall_weighted'):.6f}")

    if "balanced_accuracy_naive" in m:
        print("-" * len(title))
        print(f"balanced_accuracy_naive : {m.get('balanced_accuracy_naive'):.6f}")
        print(f"f1_macro_naive          : {m.get('f1_macro_naive'):.6f}")
        print(f"f1_weighted_naive       : {m.get('f1_weighted_naive'):.6f}")
        print(f"bal_acc_gain_vs_naive   : {m.get('balanced_accuracy_gain_vs_naive'):.6f}")
        print(f"f1_macro_gain_vs_naive  : {m.get('f1_macro_gain_vs_naive'):.6f}")
        print(f"f1_weighted_gain_vs_naive: {m.get('f1_weighted_gain_vs_naive'):.6f}")


__all__ = [
    "ClassificationMetricsConfig",
    "compute_classification_metrics",
    "metrics_to_flat_dict",
    "print_classification_report_block",
]
'''

Lo importamos así en las notebooks de entrenamiento:

```python
from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

result = compute_classification_metrics(
    y_true=y_valid,
    y_pred=y_pred_valid,
    model_name="lstm",
    split="valid",
    target="t2_dir_thr_90",
    labels=[-1, 0, 1],  # o las clases que estés usando
)

print_classification_report_block(result)

flat = metrics_to_flat_dict(result)
flat
```



# **7. Función general para salidas probabilísticas y regla de decisión**

En este stage, además del módulo de métricas de clasificación, se implementa un módulo adicional para procesar las **salidas probabilísticas** de los modelos.

A diferencia del módulo de métricas, que evalúa predicciones discretas (`y_pred`), este módulo trabaja con las **probabilidades por clase** generadas por el modelo.

## **7.1 Objetivo**

El objetivo de esta función es transformar la salida del modelo en una estructura organizada que permita:

* acceder a la probabilidad de cada clase (`-1`, `0`, `+1`)
* identificar la clase predicha (`argmax`)
* calcular la **confidence**
* preparar los datos para aplicar reglas de decisión operativa

Esto permite separar claramente:

* **evaluación estadística del modelo** (métricas)
* **uso operativo de las predicciones** (decisiones de trading)

## **7.2 Salida del modelo**

Para cada muestra \(t\), el modelo produce:

$$
P(Y=-1), \quad P(Y=0), \quad P(Y=1)
$$

A partir de estas probabilidades se derivan:

* clase predicha:
  
  $$
  \hat{y}_t = \arg\max P(Y \mid X_t)
  $$

* confidence:

  $$
  \text{confidence}_t = \max P(Y \mid X_t)
  $$

## **7.3 Estructura de salida**


La función genera una tabla (DataFrame) donde cada fila corresponde a una muestra e incluye:

* probabilidad por clase (`proba_-1`, `proba_0`, `proba_1`)
* clase predicha (`pred_label`)
* confidence
* etiqueta real (`y_true`, opcional)
* indicador de acierto (`is_correct`, opcional)

Esta estructura estandariza la salida de todos los modelos, independientemente de su tipo (tabular o secuencial).

## **7.4 Regla de decisión**


Sobre esta salida probabilística se puede aplicar una **regla de decisión** para convertir predicciones en señales operativas.

Por ejemplo:

* si `P(Y=+1) ≥ threshold_long` → **BUY**
* si `P(Y=-1) ≥ threshold_short` → **SELL**
* en otro caso → **NO TRADE**

Esto permite controlar:

* frecuencia de operación
* calidad de las señales
* trade-off entre cantidad y precisión

## **7.5 Interpretación**


Este módulo no evalúa el modelo, sino que prepara su salida para uso operativo.

La separación queda entonces definida como:

* **modelo** → estima probabilidades
* **módulo probabilístico** → organiza y transforma esas probabilidades
* **regla de decisión** → define cuándo operar

## **Idea central**

El modelo no toma decisiones de trading.

> El modelo estima probabilidades; la estrategia decide cómo usarlas.

En una frase, este módulo sirve para traducir la salida probabilística del modelo a una estructura usable para análisis operativo, sin mezclarla con las métricas de clasificación.





## **Código ejecutable**

In [11]:
classification_probabilities_py = '''
"""
classification_probabilities.py

Utilidades para procesar salidas probabilísticas de modelos de clasificación
multiclase seq2one aplicados al proyecto MNQ T2.

Diseñado para ser importado desde notebooks o scripts de evaluación.

Objetivo:
- organizar probabilidades por clase
- obtener clase predicha
- calcular confidence
- preparar la salida para reglas de decisión operativa posteriores

Autor: OpenAI / Proyecto MNQ
"""

from __future__ import annotations

from typing import Any, Dict, Iterable, Optional

import numpy as np
import pandas as pd


def _to_1d_numpy(y: Iterable[Any], name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray 1D.

    Parámetros
    ----------
    y : iterable
        Vector de etiquetas.
    name : str
        Nombre de la variable para mensajes de error.

    Retorna
    -------
    np.ndarray
        Vector 1D.
    """
    arr = np.asarray(y)

    if arr.ndim == 0:
        raise ValueError(f"{name} no puede ser escalar; se esperaba un vector 1D.")
    if arr.ndim > 1:
        arr = arr.reshape(-1)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío.")

    return arr


def _validate_class_labels(class_labels: Iterable[Any]) -> np.ndarray:
    """
    Valida el vector de clases.

    Parámetros
    ----------
    class_labels : iterable
        Clases en el mismo orden que las columnas de y_proba.

    Retorna
    -------
    np.ndarray
        Vector validado de clases.
    """
    labels = np.asarray(list(class_labels))

    if labels.ndim != 1:
        raise ValueError("class_labels debe ser un vector 1D.")
    if labels.size == 0:
        raise ValueError("class_labels no puede estar vacío.")

    unique_labels = np.unique(labels)
    if unique_labels.size != labels.size:
        raise ValueError(
            "class_labels contiene valores duplicados. "
            "Debe especificarse un conjunto de clases sin repeticiones."
        )

    return labels


def _validate_y_proba(y_proba: Any) -> np.ndarray:
    """
    Valida la matriz de probabilidades.

    Parámetros
    ----------
    y_proba : array-like
        Matriz de probabilidades con forma (n_samples, n_classes).

    Retorna
    -------
    np.ndarray
        Matriz validada de probabilidades.
    """
    arr = np.asarray(y_proba, dtype=float)

    if arr.ndim != 2:
        raise ValueError(
            "y_proba debe ser una matriz 2D con forma (n_samples, n_classes)."
        )

    if arr.shape[0] == 0:
        raise ValueError("y_proba no contiene muestras.")
    if arr.shape[1] == 0:
        raise ValueError("y_proba no contiene clases.")

    if np.isnan(arr).any():
        raise ValueError("y_proba contiene valores NaN.")

    if np.any(arr < 0.0) or np.any(arr > 1.0):
        raise ValueError("y_proba debe contener probabilidades en el rango [0, 1].")

    row_sums = arr.sum(axis=1)
    if not np.allclose(row_sums, 1.0, atol=1e-6):
        raise ValueError(
            "Cada fila de y_proba debe sumar 1. "
            "Se detectaron filas con suma distinta de 1."
        )

    return arr


def compute_probabilistic_outputs(
    y_proba: Any,
    class_labels: Iterable[Any],
    *,
    y_true: Optional[Iterable[Any]] = None,
) -> pd.DataFrame:
    """
    Procesa la salida probabilística de un clasificador multiclase seq2one.

    Parámetros
    ----------
    y_proba : array-like
        Matriz de probabilidades con forma (n_samples, n_classes).
        Cada fila representa una muestra y cada columna una clase.
    class_labels : iterable
        Clases en el mismo orden que las columnas de y_proba.
        Ejemplo: [-1, 0, 1]
    y_true : iterable, opcional
        Etiquetas reales. Si se provee, se agregan columnas de comparación.

    Retorna
    -------
    pd.DataFrame
        DataFrame con:
        - probabilidad por clase
        - clase predicha
        - confidence
        - y_true (opcional)
        - is_correct (opcional)
    """
    proba = _validate_y_proba(y_proba)
    labels = _validate_class_labels(class_labels)

    n_samples, n_classes = proba.shape

    if len(labels) != n_classes:
        raise ValueError(
            "La cantidad de class_labels debe coincidir con el número de columnas de y_proba. "
            f"Recibido: {len(labels)} labels vs {n_classes} columnas."
        )

    pred_idx = np.argmax(proba, axis=1)
    pred_labels = labels[pred_idx]
    confidence = np.max(proba, axis=1)

    data: Dict[str, Any] = {}

    for i, cls in enumerate(labels):
        data[f"proba_{cls}"] = proba[:, i]

    data["pred_label"] = pred_labels
    data["confidence"] = confidence

    result = pd.DataFrame(data)

    if y_true is not None:
        y_true_arr = _to_1d_numpy(y_true, "y_true")

        if len(y_true_arr) != n_samples:
            raise ValueError(
                "y_true debe tener la misma longitud que la cantidad de muestras en y_proba. "
                f"Recibido: {len(y_true_arr)} vs {n_samples}"
            )

        result["y_true"] = y_true_arr
        result["is_correct"] = result["pred_label"].to_numpy() == y_true_arr

    return result


def apply_decision_rule(
    proba_df: pd.DataFrame,
    *,
    long_class: Any = 1,
    short_class: Any = -1,
    long_threshold: float = 0.30,
    short_threshold: float = 0.30,
) -> pd.DataFrame:
    """
    Aplica una regla de decisión operativa simple sobre probabilidades por clase.

    Parámetros
    ----------
    proba_df : pd.DataFrame
        Salida de compute_probabilistic_outputs.
    long_class : Any
        Clase asociada a señal long.
    short_class : Any
        Clase asociada a señal short.
    long_threshold : float
        Umbral mínimo para activar señal BUY.
    short_threshold : float
        Umbral mínimo para activar señal SELL.

    Retorna
    -------
    pd.DataFrame
        Copia del DataFrame con columnas adicionales:
        - signal_raw
        - trade
    """
    if not (0.0 <= long_threshold <= 1.0):
        raise ValueError("long_threshold debe estar en el rango [0, 1].")
    if not (0.0 <= short_threshold <= 1.0):
        raise ValueError("short_threshold debe estar en el rango [0, 1].")

    df = proba_df.copy()

    long_col = f"proba_{long_class}"
    short_col = f"proba_{short_class}"

    if long_col not in df.columns:
        raise ValueError(f"No existe la columna '{long_col}' en proba_df.")
    if short_col not in df.columns:
        raise ValueError(f"No existe la columna '{short_col}' en proba_df.")

    signal_raw = np.where(
        df[long_col].to_numpy() >= long_threshold,
        1,
        np.where(df[short_col].to_numpy() >= short_threshold, -1, 0),
    )

    df["signal_raw"] = signal_raw
    df["trade"] = df["signal_raw"] != 0

    return df


__all__ = [
    "compute_probabilistic_outputs",
    "apply_decision_rule",
]
'''

Ejemplo de uso:



```python
from classification_probabilities import (
    compute_probabilistic_outputs,
    apply_decision_rule,
)

proba_df = compute_probabilistic_outputs(
    y_proba=y_proba,
    class_labels=[-1, 0, 1],
    y_true=y_test,
)

signals_df = apply_decision_rule(
    proba_df,
    long_class=1,
    short_class=-1,
    long_threshold=0.30,
    short_threshold=0.30,
)
```



# **8. Guardado de ejecutables**

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from pathlib import Path
drive_path = Path("/content/drive/MyDrive/neural_profit")
path_metrics = drive_path / "metrics"
path_metrics.mkdir(parents=True, exist_ok=True)

In [9]:
classification_metrics_py_path = path_metrics / "classification_metrics.py"
classification_metrics_py_path.write_text(classification_metrics_py)
print(f"Script guardado en: {classification_metrics_py_path}")

Script guardado en: /content/drive/MyDrive/neural_profit/metrics/classification_metrics.py


In [12]:
classification_probabilities_py_path = path_metrics / "classification_probabilities.py"
classification_probabilities_py_path.write_text(classification_probabilities_py)
print(f"Script guardado en: {classification_probabilities_py_path}")

Script guardado en: /content/drive/MyDrive/neural_profit/metrics/classification_probabilities.py


## **Como importarlos en notebooks**

```python
# =========================================
# IMPORTAR MÓDULOS DESDE GOOGLE DRIVE
# =========================================

import sys
from pathlib import Path

BASE_PATH = Path("/content/drive/MyDrive/neural_profit")

# agregar paths
sys.path.append(str(BASE_PATH / "metrics"))

# importar funciones
from classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

from classification_probabilities import (
    compute_probabilistic_outputs,
    apply_decision_rule,
)

print("Módulos importados correctamente")
```

## **Uso típico (clasificación + métricas)**



```python
# =========================================
# MÉTRICAS DE CLASIFICACIÓN
# =========================================

metrics_result = compute_classification_metrics(
    y_true=y_test,
    y_pred=y_pred,
    model_name="xgboost",
    split="test",
    target="t2_p40_h30",
)

# ver reporte
print_classification_report_block(metrics_result)

# convertir a dict plano (para DataFrame)
metrics_flat = metrics_to_flat_dict(metrics_result)
```

## **Uso típico (probabilidades)**

```python
# =========================================
# SALIDA PROBABILÍSTICA
# =========================================

proba_df = compute_probabilistic_outputs(
    y_proba=y_proba,
    class_labels=[-1, 0, 1],
    y_true=y_test,   # opcional
)

proba_df.head()
```



##**Aplicar regla de decisión**

```python
# =========================================
# REGLA DE DECISIÓN OPERATIVA
# =========================================

signals_df = apply_decision_rule(
    proba_df,
    long_class=1,
    short_class=-1,
    long_threshold=0.30,
    short_threshold=0.30,
)

signals_df.head()
```

## **Ejemplo completo típico**

```python
# =========================================
# PIPELINE COMPLETO
# =========================================

# 1. métricas del modelo
metrics_result = compute_classification_metrics(
    y_true=y_test,
    y_pred=y_pred,
    model_name="xgboost",
    split="test",
    target="t2_p40_h30",
)

print_classification_report_block(metrics_result)

# 2. probabilidades
proba_df = compute_probabilistic_outputs(
    y_proba=y_proba,
    class_labels=[-1, 0, 1],
    y_true=y_test,
)

# 3. señales operativas
signals_df = apply_decision_rule(
    proba_df,
    long_threshold=0.30,
    short_threshold=0.30,
)

print(signals_df.head())
```